## Installing Chromadb and langchain_community in myDrive so that multiple time installation can be stoped.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# !pip install chromadb -t "/content/drive/MyDrive/chromadb"

In [ ]:
# !pip install langchain_community -t "/content/drive/MyDrive/langchain_community"

In [ ]:
# !pip install langchain_groq -t "/content/drive/MyDrive/langchain_groq"

In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/chromadb")
import chromadb
sys.path.append("/content/drive/MyDrive/langchain_community")
import langchain_community
sys.path.append("/content/drive/MyDrive/langchain_groq")
import langchain_groq

In [ ]:
client = chromadb.Client()
collection = client.create_collection(name = 'my_collection')

In [ ]:
collection.add(
    documents=[
        'This document is about New York',
        'This document is about Delhi'
    ],
    ids = ['id1','id2'],
    metadatas=[
        {'url':'https://en.wikipedia.org/wiki/New_york'},
        {'url':'https://en.wikipedia.org/wiki/delhi'}
    ]
)

In [ ]:
all_doc = collection.get()
all_doc1 = collection.get(ids=['id1'])
all_doc

{'ids': ['id1', 'id2'],
 'embeddings': None,
 'documents': ['This document is about New York',
  'This document is about Delhi'],
 'uris': None,
 'data': None,
 'metadatas': [{'url': 'https://en.wikipedia.org/wiki/New_york'},
  {'url': 'https://en.wikipedia.org/wiki/delhi'}],
 'included': [<IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

In [ ]:
results = collection.query(
    query_texts=['Query is about devil'],
    n_results=2
)
results

{'ids': [['id1', 'id2']],
 'embeddings': None,
 'documents': [['This document is about New York',
   'This document is about Delhi']],
 'uris': None,
 'data': None,
 'metadatas': [[{'url': 'https://en.wikipedia.org/wiki/New_york'},
   {'url': 'https://en.wikipedia.org/wiki/delhi'}]],
 'distances': [[1.652240514755249, 1.6925829648971558]],
 'included': [<IncludeEnum.distances: 'distances'>,
  <IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

In [ ]:
collection.delete(ids = all_doc['ids'])
collection.get()

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'data': None,
 'metadatas': [],
 'included': [<IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

## Web Scraping

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.wgu.edu/blog/types-jobs-computer-engineers-this-year2207.html")
page_Data = loader.load().pop().page_content
# print(page_Data)
len(page_Data)

21749

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    groq_api_key = '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'
    # other params...
)
response = llm.invoke('what is llm?')
response

AIMessage(content="LLM stands for Large Language Model. It's a type of artificial intelligence (AI) designed to process and generate human-like language. LLMs are trained on vast amounts of text data, which enables them to learn patterns, relationships, and context within language.\n\nLarge Language Models are typically characterized by their ability to:\n\n1. **Understand natural language**: LLMs can comprehend and interpret human language, including nuances, idioms, and context.\n2. **Generate text**: LLMs can create coherent and contextually relevant text, such as responses to questions, summaries of articles, or even entire articles.\n3. **Answer questions**: LLMs can provide accurate and informative answers to a wide range of questions, from simple queries to complex, open-ended questions.\n4. **Translate languages**: LLMs can translate text from one language to another, often with high accuracy.\n5. **Summarize content**: LLMs can condense long pieces of text into shorter, more d

In [ ]:
from langchain_core.prompts import PromptTemplate
prompt_extract = PromptTemplate.from_template(
    """
    ### SCRAPED TEXT FROM WEBSITE:
    {page_Data}
    ### INSTRUCTION:
    The Scraped text is about the jobsector of cse student. Now give me a list of job with the benifit of the jobs.
    Only return the valid JSON.
    ### VALID JSON (NO PREAMBLE)
    """
)
chain_extract = prompt_extract | llm
res = chain_extract.invoke(input={'page_Data':page_Data})
type(res.content)

str

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

[{'Job': 'Computer Systems Analyst',
  'Benefits': ['Median income: $99,270',
   'Growth rate: 7%',
   'High job satisfaction due to low-stress work environment, good work-life balance, and high salary potential']},
 {'Job': 'Computer Programmer',
  'Benefits': ['Median salary: $93,000',
   'Job growth: Not rapid, but generally happy with daily tasks and salary']},
 {'Job': 'Machine Learning Engineer',
  'Benefits': ['Average salary: $111,505',
   'Fourth fastest growing career in the United States']},
 {'Job': 'Web Developer',
  'Benefits': ['Median annual salary: $77,200',
   'Projected growth rate: 13%',
   'Generally low-stress environment, good job outlook, and high salary and raises throughout the career']},
 {'Job': 'Information Security Analyst',
  'Benefits': ['Median annual income: $102,600',
   'Projected growth rate: 33%',
   'Higher than the median salary for similar careers']},
 {'Job': 'Computer Forensics Analyst',
  'Benefits': ['Average annual salary: $75,018',
   'Pro